# ВКР

Интерактивный анализ; корень репозитория в `sys.path` — первая кодовая ячейка ниже.

In [ ]:
# ─── Настройка путей ──────────────────────────────────────────────────────────
import sys, os
# Добавляем корень репозитория в путь
sys.path.insert(0, os.path.abspath(".."))

import logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")


## 1. Конфигурация

Измените пути и горизонты при необходимости.

In [ ]:
from config import CFG

CFG["FILE_PATH"] = "../OZON_combined.csv"      # путь к CSV-файлу с котировками
CFG["HORIZONS"]  = [1, 5, 10]                  # горизонты прогнозирования (торг. дни)
CFG["OUT_DIR"]   = "../results"                # директория для артефактов

print("CFG загружен. Горизонты:", CFG["HORIZONS"])


## 2. Smoke Test

Быстрая проверка загрузки и формата данных.

In [ ]:
from main import smoke_test
smoke_test(CFG["FILE_PATH"])


## 3. Полный эксперимент

Цепочка как в `pipeline.run_experiment`: котировки → признаки → (опц.) новости → EDA → ARIMA–GARCH по `ARIMA_FORECAST_HORIZONS` → ML (walk-forward по `HORIZONS`).

`run_experiment` возвращает `data`, `arima_garch`, `ml`, `out_dir`. Имена файлов в `OUT_DIR` и смысл ключей — в `docs/MODEL_OVERVIEW.md`.

In [ ]:
from pipeline import run_experiment

results = run_experiment(CFG, CFG["FILE_PATH"], CFG["OUT_DIR"])
ml = results["ml"]


## 4. Walk-Forward: регрессия

In [ ]:
import pandas as pd

_reg_parts = []
for h, df in ml["wf_reg"].items():
    if df is not None and not df.empty:
        _reg_parts.append(df)
df_reg = pd.concat(_reg_parts, ignore_index=True) if _reg_parts else pd.DataFrame()

if not df_reg.empty:
    cols = [c for c in ["MAE", "RMSE", "MAPE", "MDA_%", "R2"] if c in df_reg.columns]
    display(df_reg.groupby(["model", "horizon"])[cols].mean().round(4))
else:
    print("Нет данных wf_reg.")


## 5. Walk-Forward: классификация

In [ ]:
_clf_parts = []
for h, df in ml["wf_clf"].items():
    if df is not None and not df.empty:
        _clf_parts.append(df)
df_clf = pd.concat(_clf_parts, ignore_index=True) if _clf_parts else pd.DataFrame()

if not df_clf.empty:
    cols = [c for c in ["Accuracy", "F1", "AUC"] if c in df_clf.columns]
    display(df_clf.groupby(["model", "horizon"])[cols].mean().round(4))
else:
    print("Нет данных wf_clf.")


## 6. Бенчмарки: Buy & Hold и Naive

In [ ]:
for h in CFG.get("HORIZONS", [1, 5, 10]):
    print(f"--- horizon h={h} ---")
    print("Naive(0):", ml["baseline"].get(h))
    print("Buy & Hold:", ml["buy_hold"].get(h))
    if ml["trading"].get(h):
        print("Trading (лучший clf):", ml["trading"][h])
    print("Stacking:", ml["stacking"].get(h))
    print()


## 7. Предсказания и диагностика остатков

In [ ]:
_h = CFG["HORIZONS"][0] if CFG.get("HORIZONS") else 1
preds = ml["wf_preds"].get(_h)
if preds is not None and not preds.empty:
    display(preds.head(10))
    print(f"Предсказаний (h={_h}): {len(preds)} строк")
else:
    print("Предсказания недоступны.")
